In [54]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer
import csv
import pandas as pd

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nasii\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [89]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [56]:
df = pd.read_csv('place_i.tsv', sep='\t')

In [57]:
df.head()

,placeName,placeTags,numPeopleVisited,numPeopleWant,placeDesc,placeShortDesc,placeNearby,placeAddress,placeAlt,placeLong,placeEditors,placePubDate,placeRelatedLists,placeRelatedPlaces,placeURL
0,45th Parallel Marker,"['location markers', 'marvelous maps and meas...",207,430,"Along Highway 41 between Oconto and Peshtigo,...",Almost half-way between the Equator and North...,"['Chateau Hutter', 'Peshtigo Fire Museum', 'W...","Peshtigo, Wisconsin, 54157 United States",44.9989,-87.8535,none,2010-03-15 00:00:00,"['Peshtigo Fire Museum', 'Chateau Hutter', 'W...",none,https://www.atlasobscura.com/places/45th-para...
1,Bise Village,"['villages', 'flora', 'roads', 'roadside attr...",67,429,Respect the local’s privacy as they go about ...,A network of sandy streets lined with verdant...,"['Okinawa Churaumi Aquarium', 'Izena Island',...",639-26 Bise Motobu Japan,26.7090,127.8800,none,2017-11-17 00:00:00,"['Okinawa Churaumi Aquarium', 'Izena Island',...",none,https://www.atlasobscura.com/places/bise-village
2,Carillon Berlin-Tiergarten,"['towers', 'music]",337,430,There are live concerts every Sunday at 3 p.m...,One of the world's heaviest musical instrumen...,"['Soviet Graffiti in the Reichstag', 'Memoria...",John-Foster-Dulles-Allee Berlin Germany,52.5177,13.3670,none,2018-08-06 00:00:00,"['Soviet Graffiti in the Reichstag', 'Memoria...",none,https://www.atlasobscura.com/places/carillon-...
3,Coney Island Mermaid Parade,"['mermaids', 'parades', 'tradition', 'festiva...",269,430,Check the parade's website for information on...,The famed amusement district marks the beginn...,"['Childs Restaurant Building', 'Coney Art Wal...","West 21st Street and Surf Avenue Brooklyn, Ne...",40.5746,-73.9876,none,2019-06-04 00:00:00,"['Childs Restaurant Building', 'Coney Art Wal...",none,https://www.atlasobscura.com/places/coney-isl...
4,Glasgow City Chambers' Staircase,"['marble', 'stairs', 'government', 'architect...",168,430,The City Chambers is a functioning government...,The ornate structure is part of a building bo...,['Unmarked Grave of Pierre Emile L'Angelier '...,"45 John St Glasgow, Scotland, G1 1LY United K...",55.8609,-4.2483,none,2019-10-01 00:00:00,"['Unmarked Grave of Pierre Emile L'Angelier',...",none,https://www.atlasobscura.com/places/glasgow-c...


In [58]:
df.columns

Index(['placeName ', ' placeTags ', ' numPeopleVisited ', ' numPeopleWant ',
       ' placeDesc ', ' placeShortDesc ', ' placeNearby ', ' placeAddress ',
       ' placeAlt ', ' placeLong ', ' placeEditors ', ' placePubDate ',
       ' placeRelatedLists ', ' placeRelatedPlaces ', ' placeURL'],
      dtype='object')

In [59]:
placeName = df['placeName '].isna().any()
placeDesc = df[' placeDesc '].isna().any()

In [60]:
print(placeName)
print(placeDesc) 

False
False


In [61]:
df['placeName ']              = df['placeName '].astype(str).str.lower()
df[' placeDesc ']             = df[' placeDesc '].astype(str).str.lower()
df[' placeURL']               = df[' placeURL'].astype(str).str.lower()  
df.head(3)

,placeName,placeTags,numPeopleVisited,numPeopleWant,placeDesc,placeShortDesc,placeNearby,placeAddress,placeAlt,placeLong,placeEditors,placePubDate,placeRelatedLists,placeRelatedPlaces,placeURL
0,45th parallel marker,"['location markers', 'marvelous maps and meas...",207,430,"along highway 41 between oconto and peshtigo,...",Almost half-way between the Equator and North...,"['Chateau Hutter', 'Peshtigo Fire Museum', 'W...","Peshtigo, Wisconsin, 54157 United States",44.9989,-87.8535,none,2010-03-15 00:00:00,"['Peshtigo Fire Museum', 'Chateau Hutter', 'W...",none,https://www.atlasobscura.com/places/45th-para...
1,bise village,"['villages', 'flora', 'roads', 'roadside attr...",67,429,respect the local’s privacy as they go about ...,A network of sandy streets lined with verdant...,"['Okinawa Churaumi Aquarium', 'Izena Island',...",639-26 Bise Motobu Japan,26.7090,127.8800,none,2017-11-17 00:00:00,"['Okinawa Churaumi Aquarium', 'Izena Island',...",none,https://www.atlasobscura.com/places/bise-village
2,carillon berlin-tiergarten,"['towers', 'music]",337,430,there are live concerts every sunday at 3 p.m...,One of the world's heaviest musical instrumen...,"['Soviet Graffiti in the Reichstag', 'Memoria...",John-Foster-Dulles-Allee Berlin Germany,52.5177,13.3670,none,2018-08-06 00:00:00,"['Soviet Graffiti in the Reichstag', 'Memoria...",none,https://www.atlasobscura.com/places/carillon-...


In [62]:
regexp = RegexpTokenizer('\w+')
df['placeName ']           = df['placeName '].apply(regexp.tokenize)
df[' placeDesc ']          = df[' placeDesc '].apply(regexp.tokenize)
df.head(3)

,placeName,placeTags,numPeopleVisited,numPeopleWant,placeDesc,placeShortDesc,placeNearby,placeAddress,placeAlt,placeLong,placeEditors,placePubDate,placeRelatedLists,placeRelatedPlaces,placeURL
0,"[45th, parallel, marker]","['location markers', 'marvelous maps and meas...",207,430,"[along, highway, 41, between, oconto, and, pes...",Almost half-way between the Equator and North...,"['Chateau Hutter', 'Peshtigo Fire Museum', 'W...","Peshtigo, Wisconsin, 54157 United States",44.9989,-87.8535,none,2010-03-15 00:00:00,"['Peshtigo Fire Museum', 'Chateau Hutter', 'W...",none,https://www.atlasobscura.com/places/45th-para...
1,"[bise, village]","['villages', 'flora', 'roads', 'roadside attr...",67,429,"[respect, the, local, s, privacy, as, they, go...",A network of sandy streets lined with verdant...,"['Okinawa Churaumi Aquarium', 'Izena Island',...",639-26 Bise Motobu Japan,26.7090,127.8800,none,2017-11-17 00:00:00,"['Okinawa Churaumi Aquarium', 'Izena Island',...",none,https://www.atlasobscura.com/places/bise-village
2,"[carillon, berlin, tiergarten]","['towers', 'music]",337,430,"[there, are, live, concerts, every, sunday, at...",One of the world's heaviest musical instrumen...,"['Soviet Graffiti in the Reichstag', 'Memoria...",John-Foster-Dulles-Allee Berlin Germany,52.5177,13.3670,none,2018-08-06 00:00:00,"['Soviet Graffiti in the Reichstag', 'Memoria...",none,https://www.atlasobscura.com/places/carillon-...


In [63]:
# Make a list of english stopwords
stopwords = nltk.corpus.stopwords.words("english")

In [64]:
df['placeName ']           = df['placeName '].apply(lambda x: [item for item in x if item not in stopwords])
df[' placeDesc ']          = df[' placeDesc '].apply(lambda x: [item for item in x if item not in stopwords])
df.head(3)

,placeName,placeTags,numPeopleVisited,numPeopleWant,placeDesc,placeShortDesc,placeNearby,placeAddress,placeAlt,placeLong,placeEditors,placePubDate,placeRelatedLists,placeRelatedPlaces,placeURL
0,"[45th, parallel, marker]","['location markers', 'marvelous maps and meas...",207,430,"[along, highway, 41, oconto, peshtigo, wiscons...",Almost half-way between the Equator and North...,"['Chateau Hutter', 'Peshtigo Fire Museum', 'W...","Peshtigo, Wisconsin, 54157 United States",44.9989,-87.8535,none,2010-03-15 00:00:00,"['Peshtigo Fire Museum', 'Chateau Hutter', 'W...",none,https://www.atlasobscura.com/places/45th-para...
1,"[bise, village]","['villages', 'flora', 'roads', 'roadside attr...",67,429,"[respect, local, privacy, go, daily, life, pla...",A network of sandy streets lined with verdant...,"['Okinawa Churaumi Aquarium', 'Izena Island',...",639-26 Bise Motobu Japan,26.7090,127.8800,none,2017-11-17 00:00:00,"['Okinawa Churaumi Aquarium', 'Izena Island',...",none,https://www.atlasobscura.com/places/bise-village
2,"[carillon, berlin, tiergarten]","['towers', 'music]",337,430,"[live, concerts, every, sunday, 3, p, may, sep...",One of the world's heaviest musical instrumen...,"['Soviet Graffiti in the Reichstag', 'Memoria...",John-Foster-Dulles-Allee Berlin Germany,52.5177,13.3670,none,2018-08-06 00:00:00,"['Soviet Graffiti in the Reichstag', 'Memoria...",none,https://www.atlasobscura.com/places/carillon-...


In [66]:
#keep only words which are longer than 2 letters and join the tokenized string
df['placeName_new ']           = df['placeName '].apply(lambda x: ' '.join([item for item in x if len(item)>2]))
df[' placeDesc_new ']          = df[' placeDesc '].apply(lambda x: ' '.join([item for item in x if len(item)>2]))

In [67]:
df[['placeName ','placeName_new ',' placeDesc ',' placeDesc_new ',' placeURL']].head()

,placeName,placeName_new,placeDesc,placeDesc_new,placeURL
0,"[45th, parallel, marker]",45th parallel marker,"[along, highway, 41, oconto, peshtigo, wiscons...",along highway oconto peshtigo wisconsin look h...,https://www.atlasobscura.com/places/45th-para...
1,"[bise, village]",bise village,"[respect, local, privacy, go, daily, life, pla...",respect local privacy daily life places okinaw...,https://www.atlasobscura.com/places/bise-village
2,"[carillon, berlin, tiergarten]",carillon berlin tiergarten,"[live, concerts, every, sunday, 3, p, may, sep...",live concerts every sunday may september featu...,https://www.atlasobscura.com/places/carillon-...
3,"[coney, island, mermaid, parade]",coney island mermaid parade,"[check, parade, website, information, year, da...",check parade website information year date reg...,https://www.atlasobscura.com/places/coney-isl...
4,"[glasgow, city, chambers, staircase]",glasgow city chambers staircase,"[city, chambers, functioning, governmental, bu...",city chambers functioning governmental buildin...,https://www.atlasobscura.com/places/glasgow-c...


In [69]:
#create list of all words in these columns
all_name   = ' '.join([word for word in df['placeName_new ']])
all_desc   = ' '.join([word for word in df[' placeDesc_new ']])

In [51]:
#all_name

In [70]:
all_desc

'along highway oconto peshtigo wisconsin look historical sign marker might miss respect local privacy daily life places okinawa best accessed car live concerts every sunday may september featuring blend popular music classic works typical carillon music check website changes schedule special holiday performances automated music plays daily noon check parade website information year date register since streets closed parade best take public transportation take train stillwell avenue city chambers functioning governmental building inform security information desk interested viewing staircases two sets staircase either side access though free limited able roam around leisure take photos examine display cases containing various objects pertaining building history gain entry upper levels suggested take one tours offered free minute tour twice day monday friday tours filled first come first served basis fit groups people tours located main entrance located george square side building free pa

In [71]:
#Tokenize all_words
nltk.download('punkt')
tokenized_name = nltk.tokenize.word_tokenize(all_name)
tokenized_desc = nltk.tokenize.word_tokenize(all_desc)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nasii\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [72]:
#Create a frequency distribution which records the number of times each word has occurred:
name_list = FreqDist(tokenized_name)
desc_list = FreqDist(tokenized_desc)

In [73]:
name_list

FreqDist({'museum': 17, 'park': 11, 'island': 6, 'house': 5, 'rock': 5, 'city': 4, 'hill': 4, 'monument': 4, 'historic': 4, 'tower': 4, ...})

In [74]:
desc_list

FreqDist({'take': 46, 'road': 45, 'open': 40, 'park': 38, 'street': 36, 'right': 35, 'walk': 31, 'located': 30, 'free': 29, 'parking': 28, ...})

In [22]:
type(name_list)

nltk.probability.FreqDist

In [75]:
ps = PorterStemmer()
  
# choose some words to be stemmed
#words = ["program", "programs", "programmer", "programming", "programmers"]
  
for w in desc_list:
    print(w, " : ", ps.stem(w))

take  :  take
road  :  road
open  :  open
park  :  park
street  :  street
right  :  right
walk  :  walk
located  :  locat
free  :  free
parking  :  park
miles  :  mile
museum  :  museum
station  :  station
turn  :  turn
left  :  left
get  :  get
also  :  also
north  :  north
one  :  one
available  :  avail
see  :  see
south  :  south
visit  :  visit
follow  :  follow
site  :  site
bus  :  bu
entrance  :  entranc
may  :  may
city  :  citi
side  :  side
tours  :  tour
trail  :  trail
place  :  place
check  :  check
lot  :  lot
hours  :  hour
website  :  websit
drive  :  drive
admission  :  admiss
public  :  public
tour  :  tour
main  :  main
center  :  center
river  :  river
town  :  town
mile  :  mile
accessible  :  access
along  :  along
best  :  best
building  :  build
around  :  around
east  :  east
area  :  area
closed  :  close
small  :  small
hill  :  hill
end  :  end
look  :  look
car  :  car
train  :  train
avenue  :  avenu
access  :  access
day  :  day
exit  :  exit
easily  :  

In [76]:
Stem_dict = dict([(y,x+1) for x,y in enumerate(sorted(set(desc_list)))])

In [77]:
print(Stem_dict)

{'0091': 1, '100': 2, '101': 3, '10am': 4, '112': 5, '115': 6, '116': 7, '120': 8, '1221': 9, '124': 10, '12e': 11, '12th': 12, '131': 13, '1415': 14, '145': 15, '146': 16, '150': 17, '15e': 18, '1615': 19, '169': 20, '16th': 21, '1748': 22, '175': 23, '1764': 24, '178': 25, '18360': 26, '185': 27, '18km': 28, '1929': 29, '1969': 30, '197': 31, '1st': 32, '200': 33, '2013': 34, '2014': 35, '2017': 36, '2018': 37, '2019': 38, '2021': 39, '208': 40, '209': 41, '20th': 42, '2100': 43, '2101': 44, '219': 45, '240': 46, '250': 47, '254': 48, '257': 49, '25th': 50, '26th': 51, '29th': 52, '2km': 53, '2nd': 54, '300': 55, '30th': 56, '310': 57, '3100': 58, '31st': 59, '3200': 60, '333': 61, '334': 62, '34th': 63, '371': 64, '3rd': 65, '401': 66, '406': 67, '4442': 68, '4pm': 69, '4th': 70, '502': 71, '5099': 72, '545': 73, '575': 74, '5pm': 75, '600': 76, '60th': 77, '615': 78, '65th': 79, '714': 80, '727': 81, '731': 82, '736': 83, '737': 84, '742': 85, '7584': 86, '760': 87, '782': 88, '7pm

In [79]:
term = {'term': []}
term_id = {'term_id' : []}

In [80]:
print(term)
print(term_id)

{'term': []}
{'term_id': []}


In [81]:
for k in Stem_dict.keys():
    term['term'].append(k)

In [82]:
print(term)

{'term': ['0091', '100', '101', '10am', '112', '115', '116', '120', '1221', '124', '12e', '12th', '131', '1415', '145', '146', '150', '15e', '1615', '169', '16th', '1748', '175', '1764', '178', '18360', '185', '18km', '1929', '1969', '197', '1st', '200', '2013', '2014', '2017', '2018', '2019', '2021', '208', '209', '20th', '2100', '2101', '219', '240', '250', '254', '257', '25th', '26th', '29th', '2km', '2nd', '300', '30th', '310', '3100', '31st', '3200', '333', '334', '34th', '371', '3rd', '401', '406', '4442', '4pm', '4th', '502', '5099', '545', '575', '5pm', '600', '60th', '615', '65th', '714', '727', '731', '736', '737', '742', '7584', '760', '782', '7pm', '7th', '80220', '827', '845', '855', '8am', '8pm', '8th', '9am', 'a450', 'a5025', 'a965', 'abandoned', 'abbey', 'able', 'access', 'accessed', 'accessibility', 'accessible', 'accompaniment', 'according', 'accordingly', 'across', 'activity', 'actual', 'actually', 'adams', 'adding', 'addition', 'address', 'adelaide', 'adjacent', 'ad

In [83]:
for v in Stem_dict.values():
    term_id['term_id'].append(v)

In [84]:
term_id

{'term_id': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  157

In [ ]:
word = {'word'}

In [34]:
# create an Empty DataFrame object
word_df = pd.DataFrame(word, columns = ['word'])
print(word_df)

           word
0          0091
1           100
2           101
3          10am
4           112
...         ...
2129     zagreb
2130  zhengzhou
2131     zocalo
2132        zoo
2133      ōsaki

[2134 rows x 1 columns]


In [35]:
term_df = pd.DataFrame(term_id, columns = ['term_id'])
print(term_df)

      term_id
0           1
1           2
2           3
3           4
4           5
...       ...
2129     2130
2130     2131
2131     2132
2132     2133
2133     2134

[2134 rows x 1 columns]


In [85]:
vocabulary = pd.concat([word_df,term_df ], axis=1)

In [86]:
vocabulary

,word,term_id
0,0091,1
1,100,2
2,101,3
3,10am,4
4,112,5
...,...,...
2129,zagreb,2130
2130,zhengzhou,2131
2131,zocalo,2132
2132,zoo,2133


In [87]:
#create a list of documents based on the placeDesc column
doc_list = df[' placeDesc '].values.tolist() 
doc_list

[['along',
  'highway',
  '41',
  'oconto',
  'peshtigo',
  'wisconsin',
  'look',
  'historical',
  'sign',
  'marker',
  'might',
  'miss'],
 ['respect',
  'local',
  'privacy',
  'go',
  'daily',
  'life',
  'places',
  'okinawa',
  'best',
  'accessed',
  'car'],
 ['live',
  'concerts',
  'every',
  'sunday',
  '3',
  'p',
  'may',
  'september',
  'featuring',
  'blend',
  'popular',
  'music',
  'classic',
  'works',
  'typical',
  'carillon',
  'music',
  'check',
  'website',
  'changes',
  'schedule',
  'special',
  'holiday',
  'performances',
  'automated',
  'music',
  'plays',
  'daily',
  'noon',
  '6',
  'p'],
 ['check',
  'parade',
  'website',
  'information',
  'year',
  'date',
  'register',
  'since',
  'streets',
  'closed',
  'parade',
  'best',
  'take',
  'public',
  'transportation',
  'take',
  'q',
  'n',
  'f',
  'train',
  'stillwell',
  'avenue'],
 ['city',
  'chambers',
  'functioning',
  'governmental',
  'building',
  'inform',
  'security',
  'informat

In [90]:
# instantiate the vectorizer object
countvectorizer = CountVectorizer(analyzer= 'word', stop_words='english')
tfidfvectorizer = TfidfVectorizer(analyzer='word',stop_words= 'english')

In [101]:
for i in range(len(doc_list)):
    count_wm = countvectorizer.fit_transform(doc_list[i])
    tfidf_wm = tfidfvectorizer.fit_transform(doc_list[i])
    count_tokens = countvectorizer.get_feature_names_out()
    tfidf_tokens = tfidfvectorizer.get_feature_names_out()

In [102]:
df_countvect = pd.DataFrame(data = count_wm.toarray(),columns = count_tokens)
df_tfidfvect = pd.DataFrame(data = tfidf_wm.toarray(),columns = tfidf_tokens)
print("Count Vectorizer\n")
print(df_countvect)
print("\nTD-IDF Vectorizer\n")
print(df_tfidfvect)

Count Vectorizer

    24  advance  appointment  collection  com  contact  email  hours  make  \
0    0        0            0           1    0        0      0      0     0   
1    0        0            0           0    0        0      0      1     0   
2    0        0            0           0    0        0      0      0     0   
3    0        0            0           0    0        0      0      0     0   
4    0        0            0           0    0        0      0      0     0   
5    0        0            0           0    0        0      0      0     0   
6    0        0            0           0    0        0      0      0     0   
7    0        0            0           0    0        0      0      0     0   
8    0        0            1           0    0        0      0      0     0   
9    0        0            0           0    0        0      0      0     1   
10   0        0            1           0    0        0      0      0     0   
11   0        0            0           0    0 